# Бейзлайн: детекция еды с YOLO11n

Проекту нужна детекция нескольких продуктов с классификацией каждого объекта. YOLO11n — компактный детектор, который обучается на подготовленной разметке YOLO. Исходные веса предварительно обучены на COCO; для распознавания 417 классов проекта требуется дообучение. [Документация модели](https://docs.ultralytics.com/models/yolo11/).

**Данные:** результат ноутбука 03 (`fpb_myfoodrepo_full`). Используем train для обучения, val для выбора лучшего checkpoint, размеченный test — только для итоговой оценки. Неразмеченный FPB test не участвует. Статистика данных и разбиение описаны в `docs/03_eda.md`.

**Метрики:** mAP50 и mAP50–95 (усреднение AP по классам и порогам IoU), precision и recall для объектов. Test не содержит 4 редких класса; его aggregate mAP не характеризует их качество. Метрики не показывают качество оценки калорийности.

Обучение может занять значительное время и потребовать GPU. YOLO11/Ultralytics распространяются по лицензии AGPL-3.0 либо коммерческой лицензии; перед применением в сервисе проверьте условия.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, shutil, time
from datetime import datetime

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'scripts/dataset_pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук из папки репозитория calorie_diary.')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import display

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(), '| MPS:', torch.backends.mps.is_available())

Matplotlib is building the font cache; this may take a moment.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/alfa/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Python: 3.10.20 | PyTorch: 2.14.1 | Ultralytics: 8.3.228
CUDA: False | MPS: True


## 1. Настройки и проверка данных

Запустите ноутбук 02 и 03 до этого этапа либо укажите путь к готовой сборке в `DATASET_DIR`. Список размеченного test обязателен. Перед обучением проверяются файлы изображений и меток, дубликаты и пересечение разбиений. После переноса сборки старые абсолютные пути в `data.yaml` могут устареть: здесь создаётся отдельный временный YAML с актуальными путями, исходная сборка не переписывается.

In [2]:
DATASET_DIR = Path(os.environ.get('CALORIE_DATASET_DIR', ROOT / 'data/processed/fpb_myfoodrepo_full')).expanduser().resolve()
INITIAL_WEIGHTS = Path(os.environ.get('CALORIE_INITIAL_WEIGHTS', ROOT / 'models/yolo11n_coco.pt')).expanduser().resolve()
# Чтобы оценить уже обученную модель, укажите путь к best.pt; None запускает обучение.
EXISTING_WEIGHTS = os.environ.get('CALORIE_EXISTING_WEIGHTS') or None
EXISTING_WEIGHTS = Path(EXISTING_WEIGHTS).expanduser().resolve() if EXISTING_WEIGHTS else None
EPOCHS = int(os.environ.get('CALORIE_EPOCHS', '1'))
IMAGE_SIZE = int(os.environ.get('CALORIE_IMAGE_SIZE', '640'))
BATCH = int(os.environ.get('CALORIE_BATCH', '8'))
WORKERS = int(os.environ.get('CALORIE_WORKERS', '4'))
SEED = 42
# CUDA при наличии; MPS на Apple Silicon, иначе CPU. При ошибках устройства можно указать 'cpu'.
DEVICE = os.environ.get('CALORIE_DEVICE', '0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR = ROOT / 'data/runs' / ('yolo11n_' + RUN_ID)
EXPORT_DIR = ROOT / 'models' / ('food_yolo11n_' + RUN_ID)

config_path = DATASET_DIR / 'data.yaml'
if not config_path.is_file():
    raise FileNotFoundError(f'Нет {config_path}. Укажите путь к готовому датасету или выполните ноутбуки 02 и 03.')
weights_to_load = EXISTING_WEIGHTS or INITIAL_WEIGHTS
if not weights_to_load.is_file():
    raise FileNotFoundError(f'Нет весов {weights_to_load}; см. инструкцию в models/README.md.')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
names = config.get('names')
if not isinstance(names, (list, dict)) or not names:
    raise ValueError('В data.yaml отсутствует список names.')
# Структура соответствует экспорту проекта: images/<split>/... и labels/<split>/....
split_files = {}
seen = {}
for split in ('train', 'val', 'test'):
    listing = DATASET_DIR / f'{split}.txt'
    if not listing.is_file() or not listing.read_text(encoding='utf-8').strip():
        raise ValueError(f'Нет непустого списка размеченной выборки: {listing}. Сначала выполните переразбиение в ноутбуке 03.')
    if split not in config:
        raise ValueError(f'В data.yaml отсутствует {split}; ожидается размеченное разбиение после ноутбука 03.')
    files = []
    for raw in listing.read_text(encoding='utf-8').splitlines():
        if not raw.strip():
            continue
        image = Path(raw.strip()).expanduser()
        image = (image if image.is_absolute() else listing.parent / image).resolve()
        try:
            relative = image.relative_to(DATASET_DIR)
        except ValueError as exc:
            raise ValueError(f'Путь выходит за пределы датасета: {image}') from exc
        if len(relative.parts) < 3 or relative.parts[0] != 'images':
            raise ValueError(f'Ожидается путь images/<split>/...: {image}')
        label = DATASET_DIR / 'labels' / Path(*relative.parts[1:]).with_suffix('.txt')
        if not image.is_file() or not label.is_file():
            raise FileNotFoundError(f'Нет изображения или YOLO-метки: {image}; {label}')
        if image in seen:
            raise ValueError(f'Изображение повторяется в {seen[image]} и {split}: {image}')
        seen[image] = split
        files.append(image)
    if not files:
        raise ValueError(f'Пустой список изображений: {listing}')
    split_files[split] = files
    config[split] = str(listing)
config['path'] = str(DATASET_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUNTIME_YAML = RUN_DIR / 'data.yaml'
RUNTIME_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Датасет:', DATASET_DIR, '| классов:', len(names), '| устройство:', DEVICE)
for split in ('train', 'val', 'test'):
    print(split, ':', len(split_files[split]), 'изображений')
print('Каталог запуска:', RUN_DIR)

Датасет: /Users/alfa/calorie_diary/data/processed/fpb_myfoodrepo_full | классов: 417 | устройство: mps
train : 52352 изображений
val : 9816 изображений
test : 3272 изображений
Каталог запуска: /Users/alfa/calorie_diary/data/runs/yolo11n_20261110_182310_642183


## 2. Обучение

Стартуем с COCO-весов, заменяя голову детектора под классы датасета. Для повторной оценки укажите путь к обученному `best.pt` в `EXISTING_WEIGHTS` или переменной `CALORIE_EXISTING_WEIGHTS`. `best.pt` выбирается по метрикам val, а не test. Каждый запуск сохраняется в отдельной папке и не перезаписывает предыдущий. На CPU начните с небольшого значения `EPOCHS` для проверки, но не представляйте такой запуск как результат полноценного обучения.

In [3]:
TRAIN_DIR = None
if EXISTING_WEIGHTS is None:
    model = YOLO(str(INITIAL_WEIGHTS))
    train_result = model.train(data=str(RUNTIME_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE,
                               batch=BATCH, workers=WORKERS, device=DEVICE, seed=SEED,
                               deterministic=True, project=str(RUN_DIR), name='train',
                               exist_ok=False, plots=True)
    TRAIN_DIR = Path(train_result.save_dir)
    BEST_WEIGHTS = TRAIN_DIR / 'weights/best.pt'
else:
    BEST_WEIGHTS = EXISTING_WEIGHTS
    print('Повторная оценка готовых весов без обучения.')
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Обучение не сохранило {BEST_WEIGHTS}')
print('Лучший checkpoint:', BEST_WEIGHTS)

New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.3.228 🚀 Python-3.10.20 torch-2.14.1 MPS (Apple M4 Pro)


engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/alfa/calorie_diary/data/runs/yolo11n_20261110_182310_642183/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/Users/alfa/calorie_diary/models/yolo11n_coco.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=1

Overriding model.yaml nc=80 with nc=417



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


... (8,328 outputs left out)

cream_double_cream_heavy_cream_45         62         62          0          0          0          0


Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 5.3ms postprocess per image


Results saved to /Users/alfa/calorie_diary/data/runs/yolo11n_20261110_182310_642183/train


Лучший checkpoint: /Users/alfa/calorie_diary/data/runs/yolo11n_20261110_182310_642183/train/weights/best.pt


## 3. Качество на отложенной выборке

Обученная модель загружается заново из `best.pt`. Оцениваем отдельно val и размеченный test с одинаковым размером изображения; точность и полнота зависят от выбранного библиотекой порога. Итоговый test не используется для подбора модели и гиперпараметров. Для редких классов с малым числом объектов оценки нестабильны.

In [4]:
best_model = YOLO(str(BEST_WEIGHTS))
def ordered_names(values):
    if isinstance(values, list):
        return values
    ordered = [values.get(i, values.get(str(i))) for i in range(len(values))]
    if any(value is None for value in ordered):
        raise ValueError('Словарь классов должен иметь последовательные ID от 0.')
    return ordered
if ordered_names(best_model.names) != ordered_names(names):
    raise ValueError('Классы весов не совпадают с data.yaml: используйте модель, обученную на этой сборке.')
reports = {}
for split in ('val', 'test'):
    result = best_model.val(data=str(RUNTIME_YAML), split=split, imgsz=IMAGE_SIZE,
                            batch=BATCH, workers=WORKERS, device=DEVICE,
                            project=str(RUN_DIR), name='eval_' + split,
                            exist_ok=False, plots=True)
    reports[split] = result
summary = pd.DataFrame({split: {'precision': float(r.box.mp), 'recall': float(r.box.mr),
                                  'mAP50': float(r.box.map50), 'mAP50-95': float(r.box.map)}
                        for split, r in reports.items()}).T
summary.index.name = 'split'
display(summary.round(4))

Ultralytics 8.3.228 🚀 Python-3.10.20 torch-2.14.1 MPS (Apple M4 Pro)


YOLO11n summary (fused): 100 layers, 2,743,639 parameters, 0 gradients, 7.2 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 594.8±674.4 MB/s, size: 219.1 KB)


val: Scanning /Users/alfa/calorie_diary/data/processed/fpb_myfoodrepo_full/labels/val.cache... 9816 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9816/9816 3.4Git/s 0.0s

val: Scanning /Users/alfa/calorie_diary/data/processed/fpb_myfoodrepo_full/labels/val.cache... 9816 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9816/9816 33.2Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 1/1227 1.5it/s 0.2s<13:14

... (1,732 outputs left out)

cream_double_cream_heavy_cream_45         18         18          0          0          0          0


Speed: 0.2ms preprocess, 0.6ms inference, 0.0ms loss, 4.1ms postprocess per image


Results saved to /Users/alfa/calorie_diary/data/runs/yolo11n_20261110_182310_642183/eval_test


,precision,recall,mAP50,mAP50-95
split,,,,
val,0.0044,0.1332,0.0068,0.0052
test,0.0060,0.1290,0.0084,0.0067


### Метрики по классам на test

В таблицу попадают только классы с истинными объектами в test. Обязательно смотрите на число объектов: высокое или низкое AP одного редкого класса не является устойчивой оценкой. Классы без объектов отдельно перечислены ниже.

In [5]:
def class_name(i):
    return names[i] if isinstance(names, list) else names.get(i, names.get(str(i), str(i)))

test_report = reports['test']
indices = [int(i) for i in test_report.box.ap_class_index]
test_classes = pd.DataFrame({'class_id': indices, 'class_name': [class_name(i) for i in indices],
                             'objects': [int(test_report.nt_per_class[i]) for i in indices],
                             'AP50': test_report.box.ap50, 'AP50-95': test_report.box.ap})
test_classes = test_classes.sort_values(['objects', 'class_id'], ascending=[False, True])
display(test_classes.head(20))
missing = sorted(set(range(len(names))) - set(indices))
print('Нет истинных объектов в test:', len(missing), [class_name(i) for i in missing])

,class_id,class_name,objects,AP50,AP50-95
135,135,water,270,0.455850,0.278555
360,364,bread_white,156,0.061658,0.049961
326,330,coffee,151,0.075957,0.062694
159,160,tomato,147,0.014675,0.010515
339,343,salad_leaf_salad_green,144,0.256943,0.186570
22,22,butter,122,0.003056,0.001691
12,12,black_tea,118,0.192509,0.162795
332,336,carrot,107,0.013115,0.009459
107,107,rice,99,0.022097,0.016761
355,359,bread_wholemeal,96,0.002865,0.001986


Нет истинных объектов в test: 4 ['zhal_zhaya', 'kaki', 'aperitif_with_alcohol_n_s_aperol_spritz', 'latte_macchiato']


## 4. Примеры предсказаний

Показаны первые четыре размеченных test-изображения и предсказанные рамки. Просмотрите также сохранённые графики и примеры из папки оценки: сходные продукты, мелкие объекты и несколько продуктов на одной тарелке особенно важны для качественной проверки. Отсутствие рамки не доказывает отсутствие продукта.

In [6]:
test_lines = (DATASET_DIR / 'test.txt').read_text(encoding='utf-8').splitlines()
examples = split_files['test'][:4]
fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 5), squeeze=False)
for ax, path in zip(axes.flat, examples):
    prediction = best_model.predict(source=str(path), imgsz=IMAGE_SIZE, conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(prediction.plot()[:, :, ::-1]); ax.set_title(path.name); ax.axis('off')
plt.tight_layout(); plt.show(); plt.close(fig)

<Figure size 2000x500 with 4 Axes>

## 5. Сохранение весов, метрик и выводы

`best.pt` и `last.pt` — обученные на данных проекта веса; они сохраняются отдельно для каждого запуска вместе с параметрами и метриками. `models/yolo11n_coco.pt` — только исходные веса. Выводы ниже формируются по фактически полученным результатам; качество распознавания необходимо дополнительно проверить на реальных фотографиях пользователей.

In [7]:
EXPORT_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(BEST_WEIGHTS, EXPORT_DIR / 'best.pt')
last = TRAIN_DIR / 'weights/last.pt' if TRAIN_DIR else None
if last and last.is_file(): shutil.copy2(last, EXPORT_DIR / 'last.pt')
summary.to_csv(EXPORT_DIR / 'metrics.csv')
test_classes.to_csv(EXPORT_DIR / 'test_per_class.csv', index=False)
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
metadata = {'dataset': str(DATASET_DIR), 'manifest_sha256': sha256(DATASET_DIR / 'manifest.jsonl') if (DATASET_DIR / 'manifest.jsonl').is_file() else None,
            'source_weights_sha256': sha256(weights_to_load), 'best_weights_sha256': sha256(EXPORT_DIR / 'best.pt'),
            'mode': 'evaluate' if EXISTING_WEIGHTS else 'train',
            'split_sha256': {split: sha256(DATASET_DIR / (split + '.txt')) for split in split_files},
            'train_images': len(split_files['train']),
            'val_images': len(split_files['val']),
            'test_images': len(split_files['test']), 'classes': len(names), 'absent_test_classes': missing,
            'epochs_requested': EPOCHS if TRAIN_DIR else None, 'image_size': IMAGE_SIZE, 'batch': BATCH, 'seed': SEED,
            'device': DEVICE, 'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
            'run_dir': str(RUN_DIR)}
(EXPORT_DIR / 'run.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Веса и результаты:', EXPORT_DIR)
print(f"На test: mAP50 = {summary.loc['test', 'mAP50']:.4f}; mAP50–95 = {summary.loc['test', 'mAP50-95']:.4f}; "
      f"precision = {summary.loc['test', 'precision']:.4f}; recall = {summary.loc['test', 'recall']:.4f}.")
print(f'В test не представлены {len(missing)} классов; их качество по этому test оценить нельзя.')
print('Бейзлайн позволяет сравнивать последующие модели; качество калорийности и поведение на реальных фото требуют отдельной проверки.')

gap = float(summary.loc['val', 'mAP50-95'] - summary.loc['test', 'mAP50-95'])
print(f'Разница mAP50–95 между val и test: {gap:+.4f}. '
      'Различие требует проверки на новой выборке; оно само по себе не доказывает переобучение.')
common = test_classes[test_classes['objects'] >= 10].sort_values('AP50-95').head(5)
if len(common):
    print('Классы с наименьшим AP50–95 среди классов с не менее чем 10 объектами на test:')
    display(common[['class_name', 'objects', 'AP50-95']].round(4))
else:
    print('В test нет классов с 10 и более объектами для отдельного сравнения.')
print('Детектор не оценивает массу порции или калорийность; для дневника нужны отдельные методы и проверка на пользовательских снимках.')


Веса и результаты: /Users/alfa/calorie_diary/models/food_yolo11n_20261110_182310_642183
На test: mAP50 = 0.0084; mAP50–95 = 0.0067; precision = 0.0060; recall = 0.1290.
В test не представлены 4 классов; их качество по этому test оценить нельзя.
Бейзлайн позволяет сравнивать последующие модели; качество калорийности и поведение на реальных фото требуют отдельной проверки.
Разница mAP50–95 между val и test: -0.0014. Различие требует проверки на новой выборке; оно само по себе не доказывает переобучение.
Классы с наименьшим AP50–95 среди классов с не менее чем 10 объектами на test:


,class_name,objects,AP50-95
60,icecream,15,0.0
388,chicken_cut_into_stripes_only_meat,13,0.0
370,sauce_pesto,13,0.0
365,pasta_twist,13,0.0
313,risotto,13,0.0


Детектор не оценивает массу порции или калорийность; для дневника нужны отдельные методы и проверка на пользовательских снимках.
